In [1]:
from __future__ import annotations

import os
from pathlib import Path

from scdata import DType, ScDataBank, launch_all
from tqdm.auto import tqdm


ROOT = Path(os.environ.get("mntwzq", "/mnt/shared-storage-user/dnacoding/wangzhongqi")) / (
    "Data/cellxgene/homo_spacian"
)
if not ROOT.exists():
    ROOT = ROOT.with_name("Homo_sapiens")

ROWCOUNT_DTYPES = {DType.U16, DType.U32}
bank = None
entries = []
skipped = []


def pick_rowcount(path: Path):
    datasets = launch_all(path)
    keys = ["X"]
    if "raw/X" in datasets:
        keys.append("raw/X")
    keys.extend(f"layers/{name}" for name in sorted(datasets.layers))

    for key in keys:
        ds = datasets[key]
        if ds.dtype in ROWCOUNT_DTYPES:
            return key, ds
    return None, None



paths = sorted(ROOT.rglob("*.zarr.zip"))
if not paths:
    raise SystemExit(f"no .zarr.zip found under {ROOT}")

bank = ScDataBank()
entries = []
skipped = []
for path in tqdm(paths, desc="register", unit="dataset"):
    try:
        key, ds = pick_rowcount(path)
        if ds is None:
            skipped.append((path, "no u16/u32 matrix"))
            continue

        did = bank.register(ds)
        entries.append(
            {
                "path": path,
                "matrix": key,
                "id": did,
                "dtype": ds.dtype.value,
                "n_obs": bank.dataset_num_cells(did),
                "n_vars": bank.dataset_num_genes(did),
            }
        )
    except Exception as err:
        skipped.append((path, repr(err)))

print(f"done registered={len(entries)} skipped={len(skipped)} root={ROOT}")



/home/wangzhongqi/Code/Project/scdata/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
register:  17%|█▋        | 107/622 [02:35<12:30,  1.46s/dataset]


KeyboardInterrupt: 